# GPU Kernel Generation — Google Colab Runner

This notebook sets up the environment to run the `llm-mlir-compiler` pipeline on Google Colab.

**What it does:**
1. Sets the Gemini API key
2. Clones the repository
3. Installs Python dependencies
4. Extracts the MLIR Python bindings tarball
5. Runs the benchmark pipeline using Gemini for LLM inference

**Prerequisites:**
- A Google Colab account (free tier works)
- Turn on GPU acceleration: Runtime → Change runtime type → Hardware accelerator: T4
- A Gemini API key from [Google AI Studio](https://aistudio.google.com/app/apikey)

In [ ]:
# Cell 1: Set Gemini API key
# Get your key from: https://aistudio.google.com/app/apikey

import os
os.environ["GEMINI_API_KEY"] = "your_gemini_api_key_here"

# Verify it's set
assert os.environ["GEMINI_API_KEY"] != "your_gemini_api_key_here", "Please replace with your actual Gemini API key!"
print("✅ Gemini API key configured.")

In [ ]:
# Cell 2: Clone the repository

import os

REPO_URL = "https://github.com/seradotcom/gpu-kernel-generation.git"  # Update if needed
REPO_DIR = "/content/gpu-kernel-generation"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}

%cd {REPO_DIR}
!ls -la

In [ ]:
# Cell 3: Install Python dependencies

!pip install -q -r requirements.txt
print("✅ Python dependencies installed.")

In [ ]:
# Cell 4: Verify MLIR bindings extract and import correctly

import sys
sys.path.insert(0, "/content/gpu-kernel-generation")

from core.config import MLIR_BINDINGS_PATH
print(f"MLIR bindings path: {MLIR_BINDINGS_PATH}")

try:
    from mlir.ir import Context
    print("✅ MLIR bindings import successful")
    MLIR_OK = True
except Exception as e:
    print(f"❌ MLIR import failed: {type(e).__name__}: {e}")
    print("\nThe tarball may need to be re-compiled for Colab's environment.")
    print("You can skip MLIR verification and rely on Pydantic + Semantic validation only.")
    MLIR_OK = False

In [ ]:
# Cell 5: Smoke test — verify all core modules import

import sys
sys.path.insert(0, "/content/gpu-kernel-generation")

from core.schemas import MlirResponse
from core.semantic_validator import SemanticValidator
from core.triton_python_generator import TritonPythonGenerator
from core.triton_executor import TritonExecutor
from core.prompt_builder import PromptBuilder

# MLIR translator depends on native bindings
if MLIR_OK:
    from core.mlir_translator import MLIRTranslator
    print("✅ MLIRTranslator imported")
else:
    print("⚠️  MLIRTranslator skipped (bindings not available)")

print("✅ All core modules imported successfully.")

In [ ]:
# Cell 6: Run the benchmark pipeline (uses Gemini API)

%cd /content/gpu-kernel-generation
!python run_benchmarks.py

In [ ]:
# Cell 7 (Optional): Test a single custom prompt interactively

import json
import sys
sys.path.insert(0, "/content/gpu-kernel-generation")

from core.llm_client import generate_llm_response
from core.schemas import MlirResponse
from core.semantic_validator import SemanticValidator
from core.triton_python_generator import TritonPythonGenerator
from core.triton_executor import TritonExecutor
from core.prompt_builder import PromptBuilder

if MLIR_OK:
    from core.mlir_translator import MLIRTranslator
    translator = MLIRTranslator()
else:
    translator = None

validator = SemanticValidator()
gen = TritonPythonGenerator()
executor = TritonExecutor()
prompt_builder = PromptBuilder()

user_prompt = "Generate a Triton kernel that performs element-wise addition of two float32 vectors."
system_prompt = prompt_builder.build_prompt(user_prompt, MlirResponse.model_json_schema())

print("Calling Gemini API...")
raw = generate_llm_response("gemini", system_prompt, user_prompt, schema=MlirResponse)
print("✅ Raw response received.")

# Clean markdown wrappers if present
clean = raw.strip()
if clean.startswith("```json"): clean = clean[7:]
if clean.startswith("```"): clean = clean[3:]
if clean.endswith("```"): clean = clean[:-3]
clean = clean.strip()

mlir_obj = MlirResponse(**json.loads(clean))
print("✅ Pydantic validation passed.")

# Semantic validation
errors = validator.validate(mlir_obj)
if errors:
    print(f"❌ Semantic errors: {errors}")
else:
    print("✅ Semantic validation passed.")

# MLIR translation (if available)
if translator:
    mlir_code = translator.translate_to_module(mlir_obj.code)
    print("✅ MLIR verification passed.")
else:
    print("⚠️  Skipping MLIR verification (bindings not available)")

# Generate Triton Python
triton_py = gen.generate(mlir_obj.code)
print("\n=== Generated Triton Python ===")
print(triton_py)

# Compile and benchmark
result = executor.run(triton_py, n_elements=1024)
print(f"\n=== Execution Result ===")
print(json.dumps(result, indent=2))

---

## Troubleshooting

**Git clone fails:** Colab sometimes blocks GitHub. Try uploading the repo as a zip file and extracting it instead.

**Gemini API rate limit:** If you get rate-limited, add `time.sleep(5)` between API calls or upgrade to a paid plan.

**MLIR bindings fail to import:** The tarball was compiled for a specific Linux environment. If it doesn't work on Colab, ask your teammate to re-compile inside a Colab notebook and replace the tarball.

**Triton compilation fails:** Make sure GPU is enabled (T4). Go to Runtime → Change runtime type → Hardware accelerator: T4.

**CUDA out of memory:** Reduce `n_elements` in `run_benchmarks.py` or the executor call.